In [2]:
!pip install -q keras-tuner

import numpy as np
import keras_tuner as kt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.utils import to_categorical

# Load Iris dataset
iris = load_iris()
X = iris.data
y = to_categorical(iris.target)

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Normalize data
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

# Build model
def build_model(hp):
    model = Sequential([
        Input(shape=(4,)),
        Dense(hp.Choice('units', [16, 32, 64]), activation='relu'),
        Dropout(hp.Choice('dropout', [0.2, 0.3, 0.5])),
        Dense(3, activation='softmax')
    ])

    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# Hyperparameter tuning
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=3,
    directory='tuning',
    project_name='iris'
)

# Search best parameters
tuner.search(
    X_train, y_train,
    epochs=10,
    validation_split=0.2
)

print("Best Parameters:")
print(tuner.get_best_hyperparameters()[0].values)

Trial 3 Complete [00h 00m 05s]
val_accuracy: 0.7083333134651184

Best val_accuracy So Far: 0.7916666865348816
Total elapsed time: 00h 00m 15s
Best Parameters:
{'units': 64, 'dropout': 0.3}
